In [ ]:
import feedparser
import requests
import urllib.parse
from dotenv import load_dotenv
import os
from langchain_mistralai import ChatMistralAI
from langchain_core.prompts import PromptTemplate
from firecrawl import Firecrawl
import time
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, HttpUrl, Field
from typing import List
from langchain_core.runnables import RunnableLambda, RunnableParallel
from langgraph.graph import StateGraph,START,END
from typing import TypedDict

In [ ]:
load_dotenv()

In [ ]:
rss_category_feeds ={
    "Sports":{
        "Football":[
                "https://feeds.bbci.co.uk/sport/football/rss.xml","https://www.espn.com/espn/rss/soccer/news",
                "https://www.goal.com/feeds/en/news","https://www.skysports.com/rss/12040"
        ],
        "Cricket":[
            "https://www.espncricinfo.com/rss/content/story/feeds/0.xml","https://feeds.bbci.co.uk/sport/cricket/rss.xml","https://www.cricbuzz.com/cricket-rss-feeds",
            "https://livemint.com/rss/sports"
        ],
        "BasketBall":[
            "https://www.espn.com/espn/rss/nba/news","https://www.cbssports.com/rss/headlines/nba"
        ],
        "Hockey":[
            "https://www.espn.com/espn/rss/nhl/news","https://www.cbssports.com/rss/headlines/nhl","https://www.nbcsports.com/nhl",
            "https://www.nhl.com/rss/news.xml"
        ]
    },
    "Tech":{
        "AI":[
            "https://www.technologyreview.com/feed/","https://venturebeat.com/category/ai/feed/",
            "https://www.techrepublic.com/rssfeeds/articles","https://spectrum.ieee.org/feeds/feed.rss"
        ],
        "Startups & Business":[
            "https://techcrunch.com/category/startups/feed/","https://venturebeat.com/feed/","https://www.geekwire.com/feed/",
            "https://www.producthunt.com/feed"
        ],
        "Mobile & Apps" :[
            "https://www.androidauthority.com/feed/","https://9to5google.com/feed/","https://9to5mac.com/feed/"
        ],
        "Cybersecurity":[
            "https://krebsonsecurity.com/feed/","https://feeds.feedburner.com/TheHackersNews","https://www.darkreading.com/rss.xml"
        ]
    },
    "Finance":{
        "Stocks & Investing":[  "https://seekingalpha.com/feed.xml"
        ],
        "Crypto & Blockchain":[
            "https://www.coindesk.com/arc/outboundfeeds/rss/","https://bitcoinmagazine.com/feed",
            "https://cointelegraph.com/rss","https://decrypt.co/feed"
        ],
        "India-Focused":[
            "https://www.moneycontrol.com/rss/latestnews.xml","https://economictimes.indiatimes.com/markets/rssfeeds/1977021501.cms",
            "https://www.livemint.com/rss/markets"
        ],
        "General Finance & Markets":[
            "https://feeds.bloomberg.com/markets/news.rss",
            "https://feeds.a.dj.com/rss/RSSMarketsMain.xml","https://www.economist.com/finance-and-economics/rss.xml"
        ]
    },
    "Science":{
        "Biology & Medicine":[
            "https://www.science.org/action/showFeed?type=axatoc&feed=rss&jc=science"
        ],
        "Space & Astronomy":[
            "https://www.nasa.gov/rss/dyn/breaking_news.rss","https://www.space.com/feeds/all",
            "https://skyandtelescope.org/feed/"
        ],
        "Physics & Chemistry":[
            "https://spectrum.ieee.org/feeds/feed.rss",

        ],
        " Environment & Climate":[
            "https://www.nationalgeographic.com/rss","https://www.climate.gov/feeds/all.rss",
            "https://www.eurekalert.org/rss/all.xml","https://www.nsf.gov/rss/rss_www_news.xml"
        ]
    }

}

In [ ]:
app = Firecrawl(api_key=os.environ.get("FIRECRAWL_API_KEY",""))

In [ ]:
class NewsItem(BaseModel):
    """Represents a single filtered news item returned by the agent."""
    title: str = Field(..., description="The attention-grabbing news headline")
    source: str = Field(..., description="The news source (e.g. bbci, espn, skysports)")
    url: str = Field(..., description="The full URL to the article")


class FilteredNewsResponse(BaseModel):
    """Top 4 filtered news items selected by the agent."""
    items: List[NewsItem] = Field(...,min_length=4,max_length=4,description="Exactly 4 most interesting news items"
    )

class SummaryStructure(BaseModel):
    """ This  is used for summary generation. """
    summary: str = Field(...,description="3-4 line summary of the given markdown text")

class FinalResultEntity(BaseModel):
    """ This is used to get final structured output"""
    category: str = Field(...,description="Category chosen by the user")
    sport : str = Field(...,description="Out of selected category which sport they prefer")
    url: HttpUrl = Field(...,description="url of the news")
    source: str =Field(...,description="Source of the news")
    summary: str = Field(...,description="summary of the news that are after filteration")


class FinalResult(BaseModel):
    details: List[FinalResultEntity]

In [ ]:
class SportState(TypedDict):
    category: str 
    sport: str 
    url:list
    config: dict
    filtered_cnt: FilteredNewsResponse

    summary : dict

    final_output: FinalResult

In [ ]:
def extract_sport_url(state:SportState):
    category=state['category']
    sport=state['sport']
    return {"url":rss_category_feeds[category][sport]}

In [ ]:
def feedparsing(url: str) -> list:
    source = url.split(".")[1]
    result = feedparser.parse(url)
    top4_results = result["entries"][0:4]

    top4_data = []
    for res in top4_results:
        title = res["title"]
        href_url = res["links"][0]["href"]
        top4_data.append({
            "title": title,
            "link": href_url
        })

    return [{"source": source, "information": top4_data}]

In [ ]:
def create_runnable_football(state:SportState):
    urls=state['url']
    src_url = []
    for url in urls:
        source = url.split(".")[1]
        src_url.append({"source": source, "url": url})

    parallel_map = {
        item["source"]: RunnableLambda(lambda x, u=item["url"]: feedparsing(u))
        for item in src_url
    }

    parallel_chain = RunnableParallel(parallel_map) 

    return {"config":parallel_chain.invoke({})}

In [ ]:
def filter_content(state:SportState) :
    config=state['config']
    
    chat_model=ChatMistralAI(model="mistral-large-2512")
    FILTER_PROMPT = """
    You are a sports news curator. Your job is to select the TOP 4 most attention-grabbing and engaging news titles from the list below.

    ## Selection Criteria (in order of priority):
    1. **Controversy or Drama** - Fines, bans, transfers, conflicts
    2. **Big Club / Star Player involvement** - Real Madrid, Chelsea, Liverpool, Man Utd,Barcelona etc.
    3. **Urgency or Breaking News** - Decisions, warnings, surprises
    4. **Emotional Hook** - Quotes, player reactions, fan-facing stories

    ## Input News List:
   {news_data}

   ## Instructions:
   - Analyze ALL titles across ALL sources
   - Pick exactly TOP 4 that will grab the most user attention
   - Avoid duplicates (same story from different sources = pick only one)
   - No explanation, no markdown.
    {format_instructions}
    """

    parser=PydanticOutputParser(pydantic_object=FilteredNewsResponse)

    final_template=PromptTemplate(
       template=FILTER_PROMPT,
       input_variables=["news_data"],
       partial_variables={"format_instructions":parser.get_format_instructions()}
    )

    chain = final_template | chat_model | parser 

    result=chain.invoke({"news_data":config})
    return {"filtered_cnt":result}

In [ ]:
def scrape_and_summarize(url,source,chain):
    res=app.scrape(url)
    summ= chain.invoke({"text":res.markdown})
    return {
        "source":source,
        "summary":summ.summary
    }

In [ ]:
def generate_summary(state:SportState):
    filtered_cnt=state['filtered_cnt']
    summ_parser = PydanticOutputParser(pydantic_object=SummaryStructure)

    chat_model=ChatMistralAI(model="mistral-large-2512")
    summ_prompt = PromptTemplate(
        template="""You are a precise summarization assistant.

        Your task:
        - Read the markdown text carefully
        - Ignore all formatting, symbols, and redundant content
        - Extract only the core, meaningful information
        - Write a clear and concise summary in exactly 2-3 sentences
        - Do NOT include opinions, filler phrases, or repetition
        - No explanation, no markdown.

        Text:
        {text} \n
        {format_instructions}
        """,
        input_variables=["text"],
        partial_variables={"format_instructions": summ_parser.get_format_instructions()}
    )

    chain = summ_prompt | chat_model | summ_parser

    parallel_map = {
        item.title: RunnableLambda(
            lambda x, u=item.url, s=item.source: scrape_and_summarize(u, s,chain)
        )
        for item in filtered_cnt.items
    }

    parallel_chain = RunnableParallel(parallel_map)

    return {'summary':parallel_chain.invoke({})}

In [ ]:
def generate_output_indesired_format(state:SportState):
    category= state['category']
    choice = state['sport']
    filtered_cnt=state['filtered_cnt']
    summary=state['summary']

    output_parser = PydanticOutputParser(pydantic_object=FinalResult)

    chat_model=ChatMistralAI(model="mistral-medium-2508")

    Prompt_template  = PromptTemplate(
        template=""" 
        You are a precise data structuring assistant.
        The user has selected the following preferences:
        - Category : {category}   (e.g. sports, tech, finance)
        - Sport    : {choice}      (e.g. cricket, nba, football)

        this can be anything according to user choice.

        Below are the filtered news articles along with their summaries.

        Filtered Articles:
        {filtered_articles}

        Summaries:
        {summaries}

        Your Task:
        - Match each article with its corresponding summary using the title as the key
        - For every article produce one entry with:
            * category  → the user's selected category
            * sport     → the user's selected sport
            * url       → the article's url
            * source    → the article's source
            * summary   → the matched summary text
        - Do NOT invent data, skip articles, or add extra commentary
        - Return ALL articles — no omissions

        {format_instructions}
        """,
        input_variables=["category", "choice", "filtered_articles", "summaries"],
        partial_variables={"format_instructions": output_parser.get_format_instructions()}
    )


    chain = Prompt_template | chat_model | output_parser

    result: FinalResult = chain.invoke({
        "category"          : category,
        "choice"            : choice,
        "filtered_articles" : filtered_cnt,
        "summaries"         : summary,
    })

    return {"final_output": result}

In [ ]:
graph = StateGraph(SportState)

In [ ]:
graph.add_node("extract_sport_url",extract_sport_url)
graph.add_node("create_runnable_football",create_runnable_football)
graph.add_node("filter_content",filter_content)
graph.add_node("generate_summary",generate_summary)
graph.add_node("generate_output_indesired_format",generate_output_indesired_format)

In [ ]:
graph.add_edge(START,"extract_sport_url")
graph.add_edge("extract_sport_url","create_runnable_football")
graph.add_edge("create_runnable_football","filter_content")
graph.add_edge("filter_content","generate_summary")
graph.add_edge("generate_summary","generate_output_indesired_format")

In [ ]:
workflow=graph.compile()

In [ ]:
workflow

In [ ]:
def run_workflow(items):
    result= workflow.invoke({"category":items['category'],"sport":items['sport']})
    return result

In [ ]:
multi_sport_category=[{"category": "Sports", "sport": "BasketBall"},{"category":"Sports","sport":"Football"},{"category": "Sports", "sport": "Cricket"}]

In [ ]:
# parallel_chain = {
#     f"{item['category']}_{item['sport']}": RunnableLambda(
#         lambda x, i=item: run_workflow(i)
#     )
#     for item in multi_sport_category 
# }

# combined_chain = RunnableParallel(parallel_chain)

# final_res = combined_chain.invoke({})

In [ ]:
# final_res['Sports_BasketBall']['final_output']

In [ ]:
res=workflow.invoke({"category":"Science","sport":"Space & Astronomy"})

In [ ]:
print(res['final_output'].details)

In [ ]:
len(res['final_output'].details)